In [1]:
pip install opencv-python numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [7]:
# Set dataset directory
dataset_path = r"C:\Users\navan\dogs-vs-cats\train\train"  # Update this with the actual path

# Image size
IMG_SIZE = (64, 64)  # Resize to 64x64 for faster computation

# Load dataset
def load_images(folder, limit=1000):
    images = []
    labels = []
    
    file_names = os.listdir(folder)  # Get all image filenames
    cat_files = [f for f in file_names if "cat" in f][:limit]
    dog_files = [f for f in file_names if "dog" in f][:limit]

    for file in cat_files:
        img = cv2.imread(os.path.join(folder, file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, IMG_SIZE)
        images.append(img)
        labels.append(0)  # Cat = 0
    
    for file in dog_files:
        img = cv2.imread(os.path.join(folder, file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, IMG_SIZE)
        images.append(img)
        labels.append(1)  # Dog = 1

    return np.array(images), np.array(labels)

# Load data
X, y = load_images(dataset_path)
print(f"Dataset loaded: {X.shape[0]} images")


Dataset loaded: 2000 images


In [9]:
# Function to extract HOG features
def extract_hog_features(images):
    hog_features = []
    for img in images:
        feature, _ = hog(img, orientations=9, pixels_per_cell=(8, 8), 
                         cells_per_block=(2, 2), visualize=True)
        hog_features.append(feature)
    return np.array(hog_features)

# Extract features
X_hog = extract_hog_features(X)
print(f"HOG feature shape: {X_hog.shape}")

HOG feature shape: (2000, 1764)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_hog, y, test_size=0.2, random_state=42)
print(f"Training Samples: {X_train.shape[0]}, Testing Samples: {X_test.shape[0]}")

Training Samples: 1600, Testing Samples: 400


In [13]:
# Train SVM model
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train, y_train)

# Predictions
y_pred = svm_model.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
print(f"SVM Model Accuracy: {accuracy * 100:.2f}%")

SVM Model Accuracy: 77.00%


In [19]:
import os
import cv2
import numpy as np
from skimage.feature import hog

# Image size (must match training size)
IMG_SIZE = (64, 64)

# Function to predict an image using trained SVM
def predict_image(image_path, model):
    # Check if file exists
    if not os.path.exists(image_path):
        print(f"Error: Image '{image_path}' not found.")
        return "Invalid Image"

    # Load image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Check if image was successfully loaded
    if img is None:
        print(f"Error: Unable to load image '{image_path}'.")
        return "Invalid Image"

    # Resize image to match training dimensions
    img = cv2.resize(img, IMG_SIZE) / 255.0

    # Extract HOG features
    feature, _ = hog(img, orientations=9, pixels_per_cell=(8, 8), 
                     cells_per_block=(2, 2), visualize=True)

    # Predict using trained SVM model
    prediction = model.predict([feature])[0]
    
    return "Dog" if prediction == 1 else "Cat"

# Example test
test_image = r"C:\Users\navan\dogs-vs-cats\train\train\cat.36.jpg"  # Update with actual image path
result = predict_image(test_image, svm_model)

# Print the prediction
print(f"Prediction: {result}")

Prediction: Cat


In [21]:
# Example test
test_image = r"C:\Users\navan\dogs-vs-cats\train\train\dog.2252.jpg"  # Update with actual image path
result = predict_image(test_image, svm_model)

# Print the prediction
print(f"Prediction: {result}")

Prediction: Dog
